# Medical Multi-Tool AI Agent — Colab (Groq / gpt-oss-20b)

This notebook reproduces the full project inline: CSV -> SQLite conversion, three DB tools (each an internal LangChain SQL agent), a Tavily web-search tool, and a main router agent built with the OpenAI Agents SDK — all running on **Groq's `openai/gpt-oss-20b`** instead of OpenAI.

Groq exposes an OpenAI-compatible API, so both LangChain (via `langchain-groq`) and the OpenAI Agents SDK (by pointing its client at Groq's base URL) can use the same `gpt-oss-20b` model.

**Before running:** get a Groq API key (https://console.groq.com/keys) and a Tavily API key (https://app.tavily.com, free tier). In Colab, store them as secrets named `GROQ_API_KEY` and `TAVILY_API_KEY` (the key icon in the left sidebar) so the cell below picks them up automatically.

In [1]:
!pip install -q openai-agents openai langchain langchain-groq langchain-community tavily-python pandas numpy sqlalchemy python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of

In [2]:
import os

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["GROQ_MODEL"] = "openai/gpt-oss-20b"
except Exception:
    os.environ["GROQ_API_KEY"] = "gsk_..."  # fallback: paste your key here
    os.environ["GROQ_MODEL"] = "openai/gpt-oss-20b"

os.environ["LLM_PROVIDER"] = "groq"

try:
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
except Exception:
    os.environ["TAVILY_API_KEY"] = "tvly-..."  # fallback: paste your key here

## 1. Get the data

**Option A (real data):** upload `heart.csv`, `cancer.csv`, `diabetes.csv` (downloaded from the Kaggle links in the assignment) using the Colab file upload UI, into a `data/` folder.

**Option B (quick demo):** generate small synthetic CSVs with the same schemas, so the rest of the notebook runs immediately.

In [3]:
import os
os.makedirs('data', exist_ok=True)
os.makedirs('db', exist_ok=True)

import numpy as np
import pandas as pd

USE_SYNTHETIC_DATA = False  # set False once you've uploaded the real Kaggle CSVs to data/

if USE_SYNTHETIC_DATA:
    rng = np.random.default_rng(42)
    n = 500

    heart = pd.DataFrame({
        'age': rng.integers(29, 78, n), 'sex': rng.integers(0, 2, n),
        'cp': rng.integers(0, 4, n), 'trestbps': rng.integers(94, 201, n),
        'chol': rng.integers(126, 565, n), 'fbs': rng.integers(0, 2, n),
        'restecg': rng.integers(0, 3, n), 'thalach': rng.integers(71, 203, n),
        'exang': rng.integers(0, 2, n), 'oldpeak': np.round(rng.uniform(0, 6.2, n), 1),
        'slope': rng.integers(0, 3, n), 'ca': rng.integers(0, 5, n),
        'thal': rng.integers(0, 4, n), 'target': rng.integers(0, 2, n),
    })
    heart.to_csv('data/heart.csv', index=False)

    cancer = pd.DataFrame({
        'Age': rng.integers(20, 90, n), 'Gender': rng.integers(0, 2, n),
        'BMI': np.round(rng.uniform(15, 40, n), 2), 'Smoking': rng.integers(0, 2, n),
        'GeneticRisk': rng.integers(0, 3, n), 'PhysicalActivity': np.round(rng.uniform(0, 10, n), 2),
        'AlcoholIntake': np.round(rng.uniform(0, 5, n), 2), 'CancerHistory': rng.integers(0, 2, n),
        'Diagnosis': rng.integers(0, 2, n),
    })
    cancer.to_csv('data/cancer.csv', index=False)

    diabetes = pd.DataFrame({
        'Pregnancies': rng.integers(0, 17, n), 'Glucose': rng.integers(44, 199, n),
        'BloodPressure': rng.integers(24, 122, n), 'SkinThickness': rng.integers(0, 99, n),
        'Insulin': rng.integers(0, 846, n), 'BMI': np.round(rng.uniform(18, 67, n), 1),
        'DiabetesPedigreeFunction': np.round(rng.uniform(0.08, 2.42, n), 3),
        'Age': rng.integers(21, 81, n), 'Outcome': rng.integers(0, 2, n),
    })
    diabetes.to_csv('data/diabetes.csv', index=False)
    print('Synthetic CSVs written to data/.')
else:
    from google.colab import files
    print('Upload heart.csv, cancer.csv, diabetes.csv (rename to exactly these names):')
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, f'data/{fname}')

Upload heart.csv, cancer.csv, diabetes.csv (rename to exactly these names):


Saving cancer.csv to cancer.csv
Saving diabetes.csv to diabetes.csv
Saving heart.csv to heart.csv


## 2. Convert CSVs to typed SQLite databases

In [4]:
import sqlite3

def pandas_dtype_to_sql(dtype):
    kind = dtype.kind
    if kind in ('i', 'u'):
        return 'INTEGER'
    if kind == 'f':
        return 'REAL'
    if kind == 'b':
        return 'BOOLEAN'
    return 'TEXT'

def convert(csv_path, db_path, table_name):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute(f'DROP TABLE IF EXISTS "{table_name}"')
    col_defs = ', '.join(f'"{c}" {pandas_dtype_to_sql(t)}' for c, t in df.dtypes.items())
    cur.execute(f'CREATE TABLE "{table_name}" ({col_defs})')
    conn.commit()
    df.to_sql(table_name, conn, if_exists='append', index=False)
    count = cur.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    conn.close()
    print(f'{csv_path} -> {db_path} :: {table_name} ({count} rows)')

convert('data/heart.csv', 'db/heart_disease.db', 'heart_disease_records')
convert('data/cancer.csv', 'db/cancer.db', 'cancer_records')
convert('data/diabetes.csv', 'db/diabetes.db', 'diabetes_records')

data/heart.csv -> db/heart_disease.db :: heart_disease_records (1025 rows)
data/cancer.csv -> db/cancer.db :: cancer_records (1500 rows)
data/diabetes.csv -> db/diabetes.db :: diabetes_records (768 rows)


## 3. Build the four tools (LangChain SQL agents on Groq + Tavily search)

`ChatGroq` from `langchain-groq` talks to Groq's OpenAI-compatible endpoint directly, so the SQL agents just need `model=os.environ['GROQ_MODEL']` and `groq_api_key=os.environ['GROQ_API_KEY']`.

In [5]:
from functools import lru_cache
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_community.agent_toolkits.sql.base import create_sql_agent
from langchain_community.utilities import SQLDatabase
from langchain_groq import ChatGroq

llm = ChatGroq(
    model=os.environ['GROQ_MODEL'],
    groq_api_key=os.environ['GROQ_API_KEY'],
    temperature=0,
)

@lru_cache(maxsize=None)
def build_sql_agent(db_path):
    db = SQLDatabase.from_uri(f'sqlite:///{db_path}')
    toolkit = SQLDatabaseToolkit(db=db, llm=llm)
    return create_sql_agent(llm=llm, toolkit=toolkit, agent_type='openai-tools', verbose=False)

def ask_database(db_path, question):
    agent = build_sql_agent(db_path)
    result = agent.invoke({'input': question})
    return result['output']

def heart_disease_db_tool(question: str) -> str:
    return ask_database('db/heart_disease.db', question)

def cancer_db_tool(question: str) -> str:
    return ask_database('db/cancer.db', question)

def diabetes_db_tool(question: str) -> str:
    return ask_database('db/diabetes.db', question)

def medical_web_search_tool(query: str) -> str:
    from tavily import TavilyClient
    client = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])
    resp = client.search(query=f'{query} (medical)', search_depth='advanced', max_results=5, include_answer=True)
    parts = [resp['answer']] if resp.get('answer') else []
    for r in resp.get('results', [])[:3]:
        parts.append(f"- {r.get('title','')}: {r.get('content','')} ({r.get('url','')})")
    return '\n\n'.join(parts) if parts else 'No relevant medical information found.'

/tmp/ipykernel_972/1453865799.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.agent_toolkits import SQLDatabaseToolkit


## 4. Main router agent (OpenAI Agents SDK, pointed at Groq)

The OpenAI Agents SDK normally talks to `api.openai.com`. Groq's endpoint is OpenAI-compatible, so we swap in a custom `AsyncOpenAI` client with `base_url` set to Groq, and switch the SDK to the Chat Completions API (Groq doesn't support OpenAI's newer Responses API). We also disable tracing, since the SDK's built-in tracing tries to upload traces to OpenAI using the same key, which fails with a Groq key.

One more Colab-specific wrinkle: `Runner.run_sync()` refuses to run inside a notebook because Jupyter/Colab cells already execute inside their own event loop. So `ask()` below is defined as `async` and uses `Runner.run()` instead — call it with `await ask(...)` (notebooks support top-level `await`).

**Model-name gotcha:** the Agents SDK treats an `openai/` prefix on a model *string* as a routing hint ('force the OpenAI provider') and strips it before calling the API. Groq's real model id is `openai/gpt-oss-20b` (slash included), so that stripping turns it into the non-existent `gpt-oss-20b` and Groq returns 404. To avoid this we build an explicit `OpenAIChatCompletionsModel` object bound to our Groq client and pass that object (not a bare string) to `Agent(model=...)`, which skips the prefix-parsing logic entirely.

In [6]:
from openai import AsyncOpenAI
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from agents import (
    Agent,
    Runner,
    function_tool,
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)

groq_client = AsyncOpenAI(
    base_url='https://api.groq.com/openai/v1',
    api_key=os.environ['GROQ_API_KEY'],
)

set_default_openai_client(groq_client)
set_default_openai_api('chat_completions')  # Groq only supports the chat completions API, not Responses
set_tracing_disabled(True)  # tracing would otherwise try to upload to OpenAI with the Groq key

@function_tool
def HeartDiseaseDBTool(question: str) -> str:
    return heart_disease_db_tool(question)

@function_tool
def CancerDBTool(question: str) -> str:
    return cancer_db_tool(question)

@function_tool
def DiabetesDBTool(question: str) -> str:
    return diabetes_db_tool(question)

@function_tool
def MedicalWebSearchTool(query: str) -> str:
    return medical_web_search_tool(query)

ROUTER_INSTRUCTIONS = '''
You are a medical multi-tool assistant with four tools: HeartDiseaseDBTool, CancerDBTool,
DiabetesDBTool (each runs SQL over its dataset), and MedicalWebSearchTool (general medical
knowledge). Use a *DBTool for statistics/counts/averages/records from a dataset. Use
MedicalWebSearchTool for definitions/symptoms/causes/treatments not tied to a dataset.
Never fabricate dataset numbers -- only report what a tool actually returned.
'''

# Wrap the model explicitly instead of passing a bare string: this bypasses the SDK's
# 'openai/' prefix-stripping routing logic (see note above), so the full Groq model id
# reaches the API unchanged.
groq_model = OpenAIChatCompletionsModel(
    model=os.environ['GROQ_MODEL'],
    openai_client=groq_client,
)

main_agent = Agent(
    name='MedicalMultiToolAgent',
    instructions=ROUTER_INSTRUCTIONS,
    tools=[HeartDiseaseDBTool, CancerDBTool, DiabetesDBTool, MedicalWebSearchTool],
    model=groq_model,
)

async def ask(question):
    # Colab/Jupyter cells already run inside an event loop, so Runner.run_sync() (which
    # tries to start its own loop) raises RuntimeError there. Use the async Runner.run()
    # instead and 'await ask(...)' from cells below (top-level await works in notebooks).
    result = await Runner.run(main_agent, question)
    return result.final_output

## 5. Try it out

In [7]:
print(await ask('What is the average cholesterol level in the heart disease dataset?'))

The average cholesterol level in the heart disease dataset is **246.0**.


In [8]:
print(await ask('What are the symptoms of type 2 diabetes?'))

**Common symptoms of type 2 diabetes**

| Symptom | What it means | Typical presentation |
|---------|---------------|----------------------|
| **Increased thirst (polydipsia)** | Your body is trying to get rid of excess glucose | Feeling dry mouth and needing to drink more often |
| **Frequent urination (polyuria)** | The kidneys excrete more glucose, pulling water with it | Urinating many times a day, especially at night |
| **Unexplained weight loss** | Glucose isn’t entering cells, so the body burns fat for energy | Gradual weight loss despite normal appetite |
| **Increased hunger (polyphagia)** | Cells lack glucose, so you feel hungry | Eating more than usual |
| **Fatigue or tiredness** | Poor glucose uptake leads to low energy | Feeling unusually exhausted, needing frequent naps |
| **Blurred or fluctuating vision** | High blood sugar can cause fluid shifts in the eye lens | Vision changes, especially after meals |
| **Slow‑healing sores or frequent infections** | High glucose 

In [9]:
print(await ask('How many records in the cancer dataset have a positive diagnosis?'))

There are **557** records in the cancer dataset that have a positive diagnosis.
